---
### 🐛 Found a bug or have a suggestion?
[Open a GitHub Issue](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues/new?template=bug_report.md) — describe the notebook name, cell number, and what went wrong.
*Search [existing issues](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues) first to avoid duplicates.*

### 💾 Save your own copy
> **File → Save a copy in Drive** — this saves a personal copy to *your* Google Drive so your work is preserved.
> You must do this before making edits, otherwise changes are lost when the session ends.

# 📊 LDA & QDA
**ISLP Chapter 4 · Pattern Recognition for the Rest of Us**

> Jump in — each section builds on the last. Run cells top to bottom with Shift+Enter.

### What you'll learn
- Linear Discriminant Analysis — decision theory approach
- Quadratic Discriminant Analysis — relaxing the equal-covariance assumption
- LDA vs Logistic Regression — when each wins
- Confusion matrix and classification metrics

### Time: ~45 minutes

In [ ]:
# ⚠️  Run this cell first — sets up data and imports
# Tip: Runtime → Run all  (runs everything top to bottom automatically)

import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f9fa',
    'axes.grid': True, 'grid.alpha': 0.4,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 11
})

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# ── Load dataset(s) ───────────────────────────────────────────────────────────
try:
    smarket = pd.read_csv('https://www.statlearning.com/s/Smarket.csv')
    print(f'✓ Smarket.csv (ISLP): {smarket.shape}')
except Exception:
    smarket = pd.read_csv('https://raw.githubusercontent.com/ladataanalytics/pattern-recognition-notebooks/main/data/Smarket.csv')
    print(f'✓ Smarket.csv (fallback): {smarket.shape}')
smarket['Direction_num'] = (smarket['Direction']=='Up').astype(int)
print(f'  Up days: {smarket["Direction_num"].mean():.1%}')
print(f'  Python {sys.version.split()[0]} | pandas {pd.__version__}')
print('✓ Setup complete')


## 📐 Part 1 — Linear Discriminant Analysis (LDA)

### How LDA sets the decision threshold

LDA assigns x to class k that maximises the **linear discriminant score**:

```
δₖ(x) = xᵀΣ⁻¹μₖ − ½μₖᵀΣ⁻¹μₖ + log(πₖ)
```

where μₖ = class mean, Σ = shared covariance, πₖ = class prior probability.

The **decision boundary** is where δ₁(x) = δ₂(x) — solving for x gives the threshold explicitly:

```
threshold = (μ₁ + μ₂)/2  −  [Σ/(μ₁−μ₂)] · log(π₁/π₂)
```

- **Equal priors** (π₁ = π₂): threshold = midpoint between class means
- **Unequal priors**: threshold shifts toward the minority class — you automatically predict the majority class more often
- **Unequal variances**: use QDA instead — LDA's shared Σ assumption breaks down

In [ ]:
# ── LDA decision threshold: derived analytically from class statistics ──────
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# Use 1D example for clarity — balance predicts default
X_1d = smarket[['Lag1']].values
y_1d = (smarket['Direction'] == 'Up').astype(int).values

# Fit LDA and extract the parameters it learned
lda_1d = LinearDiscriminantAnalysis().fit(X_1d, y_1d)

mu0   = X_1d[y_1d==0].mean()   # class mean: Down
mu1   = X_1d[y_1d==1].mean()   # class mean: Up
pi0   = (y_1d==0).mean()       # prior: P(Down)
pi1   = (y_1d==1).mean()       # prior: P(Up)
sigma2 = np.var(X_1d)          # pooled variance (shared Σ assumption)

# Analytical threshold: midpoint + prior correction
threshold_equal_prior = (mu0 + mu1) / 2
threshold_with_prior  = (mu0 + mu1) / 2 - (sigma2 / (mu1 - mu0)) * np.log(pi1/pi0)

print("LDA threshold derivation (1D: Lag1 predicts market direction)")
print(f"  Class means:    μ(Down) = {mu0:.4f}   μ(Up) = {mu1:.4f}")
print(f"  Class priors:   π(Down) = {pi0:.3f}   π(Up) = {pi1:.3f}")
print(f"  Pooled σ²:      {sigma2:.4f}")
print()
print(f"  Threshold (equal priors):  (μ₀+μ₁)/2 = {threshold_equal_prior:.6f}")
print(f"  Threshold (actual priors): corrected  = {threshold_with_prior:.6f}")
print(f"  sklearn's threshold:       {lda_1d.scalings_[0][0] * lda_1d.xbar_[0]:.6f}  ← should match")
print()

# Visualise in 1D
x_range = np.linspace(X_1d.min(), X_1d.max(), 300)
# LDA posterior probability P(Up | x)
log_ratio = (mu1 - mu0) / sigma2 * x_range - (mu1**2 - mu0**2) / (2*sigma2) + np.log(pi1/pi0)
p_up = 1 / (1 + np.exp(-log_ratio))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: class distributions + threshold
for cls, color, label in [(0,'#1e5fa8','Down'), (1,'#e85d20','Up')]:
    xi = X_1d[y_1d==cls].ravel()
    axes[0].hist(xi, bins=40, density=True, alpha=0.5, color=color, label=label)
    mu_k = xi.mean()
    axes[0].axvline(mu_k, color=color, lw=2, ls='--', alpha=0.8)

axes[0].axvline(threshold_with_prior, color='#1a7a45', lw=2.5,
                label=f'LDA threshold = {threshold_with_prior:.4f}')
axes[0].axvline(threshold_equal_prior, color='grey', lw=1.5, ls=':',
                label=f'Midpoint = {threshold_equal_prior:.4f}')
axes[0].set_xlabel('Lag1')
axes[0].set_ylabel('Density')
axes[0].set_title('LDA Threshold: Derived from Class Statistics\n'
                   'Prior correction shifts boundary from midpoint')
axes[0].legend(fontsize=8)

# Right: posterior probability curve
axes[1].plot(x_range, p_up, color='#1e5fa8', lw=2.5,
             label='P(Up | Lag1) — LDA posterior')
axes[1].axhline(0.5, color='grey', lw=1, ls='--', label='Threshold = 0.5')
axes[1].axvline(threshold_with_prior, color='#1a7a45', lw=2,
                label=f'Decision boundary x = {threshold_with_prior:.4f}')
axes[1].set_xlabel('Lag1')
axes[1].set_ylabel('Posterior P(Up | x)')
axes[1].set_title('LDA Posterior Probability\nLinear in x when Gaussians share Σ')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

print("📌 The threshold is NOT chosen arbitrarily — it falls out of Bayes' theorem.")
print("   P(k|x) ∝ P(x|k) · P(k)  →  P(x|k) Gaussian + shared Σ → linear boundary.")
print()
print("   Unequal priors shift the boundary toward the minority class:")
print(f"   Down has prior {pi0:.3f}, Up has prior {pi1:.3f}")
print(f"   → threshold shifts {threshold_with_prior - threshold_equal_prior:+.6f} from midpoint")
print()
print("   To change the threshold (e.g. for cost-sensitive classification),")
print("   use predict_proba() and apply a custom cutoff — same as logistic regression.")


In [ ]:
# Visualize LDA vs QDA decision boundaries
from sklearn.datasets import make_classification
np.random.seed(42)

# Create two scenarios
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scenario 1: Equal covariance → LDA is optimal
X1 = np.vstack([np.random.multivariate_normal([0,0], [[1,0.5],[0.5,1]], 150),
                np.random.multivariate_normal([2,2], [[1,0.5],[0.5,1]], 150)])
y1 = np.array([0]*150+[1]*150)

# Scenario 2: Unequal covariance → QDA wins
X2 = np.vstack([np.random.multivariate_normal([0,0], [[0.5,0],[0,2]], 150),
                np.random.multivariate_normal([2,2], [[2,0],[0,0.5]], 150)])
y2 = np.array([0]*150+[1]*150)

for ax, X, y, title in zip(axes, [X1,X2], [y1,y2],
                            ['Equal covariances → LDA wins', 'Unequal covariances → QDA wins']):
    lda = LinearDiscriminantAnalysis().fit(X, y)
    qda = QuadraticDiscriminantAnalysis().fit(X, y)
    
    xx, yy = np.meshgrid(np.linspace(X[:,0].min()-0.5, X[:,0].max()+0.5, 200),
                          np.linspace(X[:,1].min()-0.5, X[:,1].max()+0.5, 200))
    grid = np.c_[xx.ravel(), yy.ravel()]
    
    Z_lda = lda.predict(grid).reshape(xx.shape)
    ax.contour(xx, yy, Z_lda, colors=['#1e5fa8'], linewidths=2, linestyles='-')
    
    Z_qda = qda.predict(grid).reshape(xx.shape)
    ax.contour(xx, yy, Z_qda, colors=['#e85d20'], linewidths=2, linestyles='--')
    
    ax.scatter(X[y==0,0], X[y==0,1], color='#1e5fa8', alpha=0.4, s=15)
    ax.scatter(X[y==1,0], X[y==1,1], color='#e85d20', alpha=0.4, s=15)
    
    from matplotlib.lines import Line2D
    ax.legend([Line2D([0],[0],color='#1e5fa8',lw=2),
               Line2D([0],[0],color='#e85d20',lw=2,ls='--')],
              [f'LDA (acc={lda.score(X,y):.2f})',
               f'QDA (acc={qda.score(X,y):.2f})'], fontsize=9)
    ax.set_title(title)

plt.suptitle('LDA vs QDA Decision Boundaries', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()
print("\n📌 LDA boundary is always a straight line. QDA can curve to fit unequal covariances.")

## 🔬 Part 2 — Quadratic Discriminant Analysis (QDA)

In [ ]:
# Compare LR, LDA, QDA on Smarket
predictors = ['Lag1','Lag2','Lag3','Lag4','Lag5','Volume']
X = smarket[predictors].values
y = smarket['Direction_num'].values

# Train on 2001-2004, test on 2005 (temporal split — important for market data!)
mask_train = smarket['Year'] < 2005
X_tr, X_te = X[mask_train], X[~mask_train]
y_tr, y_te = y[mask_train], y[~mask_train]

print(f"Train: {X_tr.shape[0]} days (2001-2004)")
print(f"Test: {X_te.shape[0]} days (2005)")

models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'LDA':                 LinearDiscriminantAnalysis(),
    'QDA':                 QuadraticDiscriminantAnalysis(),
}

print("\n=== Accuracy on 2005 Test Data ===")
for name, model in models.items():
    model.fit(X_tr, y_tr)
    acc = model.score(X_te, y_te)
    print(f"  {name:<25} {acc:.4f}")

print("\n📌 All three methods are comparable here — the stock market is hard to predict!")
print("   Baseline (always predict 'Up'): {:.4f}".format(y_te.mean()))

## 📊 Part 3 — LDA vs Logistic Regression

In [ ]:
# LDA's discriminant scores — visualize the separation
lda = LinearDiscriminantAnalysis()
lda.fit(X_tr, y_tr)

scores_tr = lda.transform(X_tr)
scores_te = lda.transform(X_te)

fig, ax = plt.subplots(figsize=(9, 4))
for split, scores, y_split, label, alpha in [
    (None, scores_tr, y_tr, 'Train', 0.4),
    (None, scores_te, y_te, 'Test', 0.9)]:
    ax.hist(scores[y_split==0, 0], bins=25, alpha=alpha*0.7,
            color='#1e5fa8', density=True,
            label=f'Down ({label})' if alpha > 0.5 else None)
    ax.hist(scores[y_split==1, 0], bins=25, alpha=alpha*0.7,
            color='#e85d20', density=True,
            label=f'Up ({label})' if alpha > 0.5 else None)

ax.axvline(0, color='black', lw=1.5, ls='--', label='Decision boundary')
ax.set_xlabel('LDA Discriminant Score')
ax.set_ylabel('Density')
ax.set_title('LDA Discriminant Scores — Up vs Down Days\n(overlap = hard classification problem)')
ax.legend()
plt.tight_layout()
plt.show()
print("\n📌 Large overlap between Up/Down scores confirms stock market is hard to predict from lagged returns")

## ✅ Part 4 — When to Use LDA over Logistic Regression

**LDA is preferred when:**
- Predictors are approximately normal within each class
- Sample size is small relative to the number of predictors (n close to p)
- Classes are well-separated (logistic regression coefficients become unstable)

**Why normality + small n favours LDA:**
Logistic regression estimates β by maximum likelihood — an iterative algorithm that needs large n to converge reliably. LDA estimates class means (μₖ) and a shared covariance matrix (Σ) — far fewer parameters, much more stable with small samples.

When the normality assumption holds, LDA is the **statistically optimal** linear classifier. Logistic regression wastes information by not using the Gaussian structure.

| Situation | Preferred method |
|-----------|-----------------|
| Large n, no normality assumed | Logistic Regression |
| Small n, normality plausible | LDA |
| Classes have different covariance | QDA |
| p > n | Regularised LDA or Lasso logistic |

In [ ]:
# ── Demonstrate LDA advantage with small samples ─────────────────────────────
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold, cross_val_score

np.random.seed(42)

# Simulate data where normality HOLDS (Gaussian classes)
# Compare LDA vs LogReg at different sample sizes
n_test    = 2000
X_all, y_all = make_classification(
    n_samples=n_test + 500, n_features=10, n_informative=5,
    n_redundant=2, random_state=42)
X_te_fixed = X_all[:n_test];  y_te_fixed = y_all[:n_test]
X_pool     = X_all[n_test:];  y_pool     = y_all[n_test:]

sample_sizes = [20, 30, 50, 75, 100, 200, 500]
lda_accs, lr_accs = [], []

for n in sample_sizes:
    idx = np.random.choice(len(X_pool), n, replace=False)
    Xn, yn = X_pool[idx], y_pool[idx]
    lda_accs.append(LinearDiscriminantAnalysis().fit(Xn, yn).score(X_te_fixed, y_te_fixed))
    try:
        lr_accs.append(LogisticRegression(max_iter=1000).fit(Xn, yn).score(X_te_fixed, y_te_fixed))
    except Exception:
        lr_accs.append(np.nan)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(sample_sizes, lda_accs, 'o-', color='#1e5fa8', lw=2.5,
        markersize=7, label='LDA')
ax.plot(sample_sizes, lr_accs,  's--', color='#e85d20', lw=2.5,
        markersize=7, label='Logistic Regression')
ax.axvline(50, color='grey', lw=1, ls=':', alpha=0.7,
           label='n = 50 (LDA advantage zone)')
ax.set_xlabel('Training sample size (n)')
ax.set_ylabel('Test accuracy (n_test=2000)')
ax.set_title('LDA vs Logistic Regression — Small Sample Performance\n'
             '(Gaussian classes — LDA assumption met)')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print("LDA vs Logistic Regression accuracy by sample size:")
print(f"{'n':>6}  {'LDA':>8}  {'LogReg':>8}  {'Winner':>8}")
print("-" * 36)
for n, la, lra in zip(sample_sizes, lda_accs, lr_accs):
    winner = 'LDA ✓' if la > lra else 'LogReg' if lra > la else 'tie'
    print(f"{n:>6}  {la:>8.3f}  {lra:>8.3f}  {winner:>8}")

print()
print("📌 With small n and Gaussian classes, LDA consistently outperforms")
print("   logistic regression — it uses the distributional structure efficiently.")
print("   As n grows, the methods converge — both learn the same boundary.")


## 🔍 Part 5 — Multiclass Classification (K > 2)

**Logistic regression** can be extended to K classes via **multinomial (softmax) regression** — estimates K−1 sets of coefficients, no normality assumption.

**LDA** extends naturally to K classes by computing K discriminant functions. One model handles all classes simultaneously — this is where LDA particularly shines over logistic regression with small n.

**Comparison:**

| | Multinomial Logistic | LDA | QDA |
|---|---|---|---|
| Assumption | None | Gaussian, shared Σ | Gaussian, class Σₖ |
| Small n | Unstable | Stable ✓ | Needs n >> p |
| Boundary | Linear | Linear | Quadratic |
| K classes | Softmax | Natural | Natural |
| Interpretability | Log-odds per class | Discriminant scores | Less interpretable |

**ISLP recommendation:** Use LDA for multiclass when normality is plausible. Use multinomial logistic otherwise.

In [ ]:
# ── Multiclass LDA, QDA, and Multinomial Logistic on Iris ────────────────────
# Iris: 3 species, 4 features — classic multiclass benchmark
from sklearn.datasets import load_iris
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, confusion_matrix
from sklearn.model_selection import train_test_split

iris   = load_iris(as_frame=True)
X_iris = iris.data.values
y_iris = iris.target
names  = iris.target_names   # setosa, versicolor, virginica

X_tr_i, X_te_i, y_tr_i, y_te_i = train_test_split(
    X_iris, y_iris, test_size=0.3, random_state=42, stratify=y_iris)

models_mc = {
    'LDA':                  LinearDiscriminantAnalysis(),
    'QDA':                  QuadraticDiscriminantAnalysis(),
    'Multinomial Logistic': LogisticRegression(multi_class='multinomial',
                                                max_iter=1000, random_state=42),
}

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
print(f"{'Method':25s}  {'Test Accuracy':>14}")
print("-" * 42)
for ax, (name, model) in zip(axes, models_mc.items()):
    model.fit(X_tr_i, y_tr_i)
    y_pred_i = model.predict(X_te_i)
    acc = model.score(X_te_i, y_te_i)
    print(f"{name:25s}  {acc:>14.3f}")

    cm = confusion_matrix(y_te_i, y_pred_i)
    ConfusionMatrixDisplay(cm, display_labels=names).plot(
        ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{name}\nAccuracy = {acc:.3f}')

plt.suptitle('Multiclass Classification — Iris Dataset (3 species)',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

# LDA discriminant scores — visualise the 2D projection
lda_full = LinearDiscriminantAnalysis(n_components=2).fit(X_tr_i, y_tr_i)
Z_tr = lda_full.transform(X_tr_i)
Z_te = lda_full.transform(X_te_i)

fig, ax = plt.subplots(figsize=(7, 5))
colors = ['#1e5fa8','#e85d20','#1a7a45']
for k, (name, color) in enumerate(zip(names, colors)):
    mask = y_tr_i == k
    ax.scatter(Z_tr[mask,0], Z_tr[mask,1], color=color, alpha=0.6,
               s=30, label=name)
ax.set_xlabel('LD1'); ax.set_ylabel('LD2')
ax.set_title('LDA Discriminant Scores — Iris\n'
             'LDA projects 4D data onto the 2 best discriminating directions')
ax.legend()
plt.tight_layout()
plt.show()

print()
print("📌 LDA reduces 4 features to 2 discriminant axes (K-1 = 2 for 3 classes).")
print("   This projection maximises class separation — the LDA 'view' of the data.")
print("   With K classes, LDA always gives K-1 discriminant dimensions.")
print()
print("Full classification report — LDA:")
lda_mc = LinearDiscriminantAnalysis().fit(X_tr_i, y_tr_i)
print(classification_report(y_te_i, lda_mc.predict(X_te_i),
                              target_names=names))


## 💼 Exercise

Using the Smarket dataset, compare LDA, QDA, and Logistic Regression on predicting market direction. Use 2001-2004 as training and 2005 as test. Which model has the highest test accuracy? Report the confusion matrix for each.

In [ ]:
# ── Exercise workspace ──────────────────────────────────────────────────────
# Write your code here



In [ ]:
# @title 📝 Quiz — LDA & QDA
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What distributional assumption does LDA make about the features?
# @markdown **Q2:** What is the key difference between LDA and QDA?
# @markdown **Q3:** When does LDA outperform logistic regression?
# @markdown **Q4:** Why do we need a prior P(Y=k) in Bayes' theorem?
# @markdown **Q5:** If your data has 5 classes and 2 predictors, how many discriminant functions does LDA produce?

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

# Collect into answers dict for grading cell
answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the AI grading cell below for feedback")

In [ ]:
_NB_NAME_="LDA & QDA"
# @title 📋 Quiz Submission
# @markdown **Click ▶ Run** → copy the output box (click the copy icon in the top-left of the output box, or click ⋮ → Copy) → paste into Gemini in this Colab session.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

if "answers" not in globals():
    print("Run the quiz cell above first, then re-run this cell.")
else:
    qa = "\n".join(
        f"Q{i}: {str(v).strip() or '(no answer)'}"
        for i, (_, v) in enumerate(answers.items(), 1)
    )
    print(f"Please grade my quiz answers for the \"{_NB_TITLE}\" notebook")
    print(f"from \"Data Pattern Recognition for the Rest of Us\" (based on ISLP).")
    if GITHUB_USERNAME.strip():
        print(f"Student: @{GITHUB_USERNAME.strip()}")
    print()
    print(qa)
    print()
    print("For each question:")
    print("  1. Say CORRECT, PARTIAL, or INCORRECT")
    print("  2. Explain in 2-3 sentences why — what did I get right or wrong")
    print("  3. Give the ideal complete answer so I know exactly what was expected")
    print("  4. If I was wrong or partial, tell me the specific concept to review")
    print("     and where it appears in the notebook (e.g. Part 2, the SHAP beeswarm plot)")
    print()
    print("End with:")
    print("  - Overall grade: Excellent / Good / Needs Review / Incomplete")
    print("  - A short study plan: which questions to re-read and what to focus on")
    print()
    print("After grading, I will paste specific outputs, charts, or code blocks")
    print("from the notebook — please explain them in detail and answer follow-up questions.")


---
### 🐛 Found a bug or have a suggestion?
[Open a GitHub Issue](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues/new?template=bug_report.md) — describe the notebook name, cell number, and what went wrong.
*Search [existing issues](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues) first to avoid duplicates.*

---
*Data Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*